In [9]:
from pathlib import Path

DATASET = Path("/Users/sakshamchauhan/Downloads/SIH_Model/Osteoarthritis_Dataset")

splits = ["train", "val", "test", "auto_test"]
classes = ["0", "1", "2"]

for split in splits:
    print("\n" + "=" * 50)
    print(split.upper())
    print("=" * 50)

    total = 0

    for cls in classes:
        folder = DATASET / split / cls

        if not folder.exists():
            print(f"{cls}: MISSING")
            continue

        files = [
            f for f in folder.iterdir()
            if f.is_file()
        ]

        count = len(files)
        total += count

        print(f"Grade {cls}: {count}")

    print(f"TOTAL: {total}")


TRAIN
Grade 0: 2286
Grade 1: 2562
Grade 2: 930
TOTAL: 5778

VAL
Grade 0: 328
Grade 1: 365
Grade 2: 133
TOTAL: 826

TEST
Grade 0: 639
Grade 1: 743
Grade 2: 274
TOTAL: 1656

AUTO_TEST
Grade 0: 604
Grade 1: 678
Grade 2: 244
TOTAL: 1526


In [11]:
from pathlib import Path
import hashlib
from collections import defaultdict

DATASET = Path("/Users/sakshamchauhan/Downloads/SIH_Model/Osteoarthritis_Dataset")

splits = ["train", "val", "test", "auto_test"]

hash_locations = defaultdict(list)

for split in splits:

    split_path = DATASET / split

    for file in split_path.rglob("*"):

        if not file.is_file():
            continue

        try:
            data = file.read_bytes()
            file_hash = hashlib.sha256(data).hexdigest()

            hash_locations[file_hash].append(
                (split, str(file))
            )

        except Exception as e:
            print("Error:", file, e)


duplicates = {
    h: locations
    for h, locations in hash_locations.items()
    if len(locations) > 1
}


print("\nDuplicate files across splits:")
print("=" * 60)

for h, locations in duplicates.items():

    print("\n")

    for split, path in locations:
        print(f"{split:10} {path}")

print("\nTotal duplicate groups:", len(duplicates))


Duplicate files across splits:


train      /Users/sakshamchauhan/Downloads/SIH_Model/Osteoarthritis_Dataset/train/.DS_Store
val        /Users/sakshamchauhan/Downloads/SIH_Model/Osteoarthritis_Dataset/val/.DS_Store
test       /Users/sakshamchauhan/Downloads/SIH_Model/Osteoarthritis_Dataset/test/.DS_Store
auto_test  /Users/sakshamchauhan/Downloads/SIH_Model/Osteoarthritis_Dataset/auto_test/.DS_Store

Total duplicate groups: 1


In [15]:
from pathlib import Path
import re
from collections import defaultdict

DATASET = Path(
    "/Users/sakshamchauhan/Downloads/SIH_Model/Osteoarthritis_Dataset"
)

splits = ["train", "val", "test", "auto_test"]


def get_subject_id(filename):
    """
    Extract the 7-digit identifier.

    Examples:
        9771440R.png   -> 9771440
        9771440_2.png -> 9771440
    """

    match = re.match(r"(\d{7})", Path(filename).stem)

    if match:
        return match.group(1)

    return None


subjects = defaultdict(lambda: defaultdict(list))


for split in splits:

    split_path = DATASET / split

    for file in split_path.rglob("*"):

        if not file.is_file():
            continue

        subject_id = get_subject_id(file.name)

        if subject_id is None:
            continue

        subjects[subject_id][split].append(file.name)


# ============================================================
# Find subjects appearing in multiple splits
# ============================================================

print("=" * 70)
print("SUBJECT-LEVEL SPLIT AUDIT")
print("=" * 70)

for split_a in splits:
    for split_b in splits:

        if split_a >= split_b:
            continue

        overlap = []

        for subject_id, locations in subjects.items():

            if split_a in locations and split_b in locations:
                overlap.append(subject_id)

        print(
            f"\n{split_a} ↔ {split_b}: "
            f"{len(overlap)} overlapping subjects"
        )


SUBJECT-LEVEL SPLIT AUDIT

train ↔ val: 0 overlapping subjects

test ↔ train: 0 overlapping subjects

test ↔ val: 0 overlapping subjects

auto_test ↔ train: 0 overlapping subjects

auto_test ↔ val: 0 overlapping subjects

auto_test ↔ test: 811 overlapping subjects
